In [1]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o")

In [2]:
class State(TypedDict):

    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str


class Ingredient(BaseModel):

    name: str
    quantity: str
    unit: str


class IngredientsOutput(BaseModel):

    ingredients: List[Ingredient]

In [ ]:
def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)
    response = structured_llm.invoke(
        f"List 5-8 ingredients needed to make {state['dish']}"
    )
    return {
        "ingredients": response.ingredients,
    }


def create_recipe(state: State):
    response = llm.invoke(
        f"Write a step by step cooking instruction for {state["dish"]}, using these ingredients {state['ingredients']}",
    )
    return {
        "recipe_steps": response.content,
    }


def describe_plating(state: State):
    response = llm.invoke(
        f"Describe how to beautifully plate this dish {state["dish"]} based on this recipe {state["recipe_steps"]}"
    )
    return {
        "plating_instructions": response.content,
    }
def gate(state:State):
    ingredients = state["ingredients"]

    if len(ingredients) > 8 or len(ingredients) < 3:
        return False
    return True

In [ ]:
from langchain_core.runnables.config import P


graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_conditional_edges(
    "list_ingredients",
    gate,
    {
        True: "create_recipe",
        False: "list_ingredients",
    }
)
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [5]:
graph.invoke({"dish": "hummus"})

/Users/bitnoorilee/projects/AiAgents/workflow-architectures/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=IngredientsOutput(ingredi...', unit='tablespoons')]), input_type=IngredientsOutput])
  return self.__pydantic_serializer__.to_python(


{'dish': 'hummus',
 'ingredients': [Ingredient(name='Chickpeas', quantity='1', unit='can (15 oz)'),
  Ingredient(name='Tahini', quantity='1/4', unit='cup'),
  Ingredient(name='Lemon juice', quantity='2-3', unit='tablespoons'),
  Ingredient(name='Garlic', quantity='1-2', unit='cloves'),
  Ingredient(name='Olive oil', quantity='2', unit='tablespoons'),
  Ingredient(name='Salt', quantity='1/2', unit='teaspoon'),
  Ingredient(name='Ground cumin', quantity='1/2', unit='teaspoon'),
  Ingredient(name='Water', quantity='2-3', unit='tablespoons')],
 'recipe_steps': "Here's a simple step-by-step guide to making delicious hummus using the ingredients listed:\n\n### Ingredients:\n- **Chickpeas**: 1 can (15 oz)\n- **Tahini**: 1/4 cup\n- **Lemon juice**: 2-3 tablespoons\n- **Garlic**: 1-2 cloves\n- **Olive oil**: 2 tablespoons\n- **Salt**: 1/2 teaspoon\n- **Ground cumin**: 1/2 teaspoon\n- **Water**: 2-3 tablespoons\n\n### Instructions:\n\n1. **Prepare Chickpeas**:\n   - Open the can of chickpeas and